# Chapter 01: Elementary Differential Geometry

Source span: printed pp. 1-24; physical DjVu pages 10-33.

This opening chapter supplies the differential-geometric grammar used everywhere else in the course. The book moves through manifolds, tangent spaces, vector and tensor fields, submanifolds, metrics, affine connections, flatness, autoparallel submanifolds, projection of connections, embedding curvature, and the Riemannian connection. In this notebook we make that vocabulary inspectable before any probability model appears. The goal is not to replace the definitions with pictures, but to let the definitions leave visible numerical traces: a tangent vector differentiates a curve through a chart, a metric turns a tangent displacement into a length, and a connection decides which acceleration counts as intrinsic.

Translation guide. A chart becomes an array of coordinates. A tangent vector becomes a derivative of a coordinate curve. A tensor field becomes a rule that consumes vectors and returns numbers or other vectors. A metric becomes a positive definite matrix at each point. An affine connection becomes a recipe for comparing tangent vectors at nearby points, encoded locally by Christoffel symbols. A curve is autoparallel when its coordinate acceleration is exactly canceled by the connection term. Curvature is the obstruction to making that cancellation globally path independent.

The visual sequence is deliberately modest: first a chart with tangent frames, then a metric ellipse in polar coordinates, then a comparison between a flat coordinate straight line and a curved embedded surface. Inspect how the same coordinate move can have different metric length, and how an apparently curved trace can still be straight for a chosen connection.


## Source-Specific Study Notes

The source chapter is deliberately foundational, so use this notebook as a calibration device for the rest of the course. When a later chapter says that an exponential family is flat, it is making a statement about the relevant affine connection, not about a Euclidean picture. When it says that a submodel has embedding curvature, it is separating intrinsic coordinates from the ambient probability space. The tangent-frame artifact should therefore be read as a warning: an embedded drawing can include directions that are not tangent to the model. The polar-metric artifact gives the second warning: even when coordinates are valid, equal coordinate increments need not have equal geometric length. The cylinder artifact gives the third warning: visible bending in an ambient space is not the same thing as intrinsic curvature.

A good self-check is to explain each later statistical object using this chapter's vocabulary. Scores are tangent vectors. Fisher information is a metric. Alpha-connections are affine connections. Exponential and mixture families are autoparallel for different connections. Curved exponential families are submanifolds whose ambient straightness and intrinsic straightness do not agree. The final symbolic Christoffel check is intentionally small because the purpose is method: derive connection coefficients from a metric, then ask what acceleration means in those coordinates.


In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

BOOK_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Methods of Information Geometry.djvu").exists())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, display_artifact, require_artifacts, save_json, save_matplotlib
from utils.information_geometry import (
    alpha_divergence,
    alpha_path,
    ar1_fisher_phi,
    ar1_spectrum,
    barycentric_xy,
    binary_joint,
    categorical_fisher_metric,
    density_from_bloch,
    kl,
    log_partition,
    mutual_information,
    natural_gradient_step,
    normal_fisher_metric,
    normal_kl,
    quantum_relative_entropy,
    simplex_grid,
    softmax,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

chapter = "chapter-01"


## Tangent Vectors Are Derivations You Can Draw

The tangent space at a point is not the paper around the point. It is the space of first-order probes through the point. The picture below uses a parabola as a one-dimensional manifold inside the plane. The arrows are not decorative normals and tangents; they encode the derivative of the chart map and the complementary direction that the embedding sees but the intrinsic manifold does not. Notice that a tangent vector can be represented in the ambient plane, yet its intrinsic coordinate is only one number.


In [ ]:
u = np.linspace(-2.0, 2.0, 250)
x = u
y = 0.28 * u**2 - 0.15
fig, ax = plt.subplots()
ax.plot(x, y, color="#1f77b4", lw=2.5, label="embedded chart image")
for u0 in [-1.4, -0.35, 0.9, 1.55]:
    p = np.array([u0, 0.28 * u0**2 - 0.15])
    tangent = np.array([1.0, 0.56 * u0])
    tangent = tangent / np.linalg.norm(tangent)
    normal = np.array([-tangent[1], tangent[0]])
    ax.arrow(*p, *(0.32 * tangent), head_width=0.035, length_includes_head=True, color="#2ca02c")
    ax.arrow(*p, *(0.22 * normal), head_width=0.03, length_includes_head=True, color="#d62728", alpha=0.75)
ax.text(-1.95, 0.72, "green: tangent derivations\nred: ambient normal directions", va="top")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("ambient x")
ax.set_ylabel("ambient y")
ax.legend(loc="lower center")
fig1 = save_matplotlib(fig, chapter, "tangent-frame-chart.png")
display_artifact(fig1)


## A Metric Makes Coordinate Steps Unequal

A Riemannian metric is the local instrument for measuring tangent displacements. In polar coordinates on the Euclidean plane the metric matrix is diagonal with entries `1` and `r^2`. That simple matrix says a unit angular change is cheap near the origin and expensive far away. The Christoffel symbols below are computed from the metric, not memorized. The two nonzero patterns we check are the signs that polar coordinate basis vectors rotate as the point moves.


In [ ]:
import sympy as sp

r, th = sp.symbols("r th", positive=True)
g = sp.Matrix([[1, 0], [0, r**2]])
g_inv = g.inv()
coords = [r, th]
Gamma = {}
for k in range(2):
    for i in range(2):
        for j in range(2):
            expr = sp.Rational(1, 2) * sum(
                g_inv[k, ell] * (sp.diff(g[ell, j], coords[i]) + sp.diff(g[ell, i], coords[j]) - sp.diff(g[i, j], coords[ell]))
                for ell in range(2)
            )
            Gamma[(k, i, j)] = sp.simplify(expr)
assert Gamma[(0, 1, 1)] == -r
assert Gamma[(1, 0, 1)] == 1 / r
assert Gamma[(1, 1, 0)] == 1 / r

fig, ax = plt.subplots()
for radius, color in [(0.55, "#1f77b4"), (1.1, "#ff7f0e"), (1.65, "#2ca02c")]:
    t = np.linspace(0, 2 * np.pi, 200)
    dr = 0.16 * np.cos(t)
    dtheta = 0.16 * np.sin(t) / radius
    ax.plot(radius + dr, dtheta, color=color, label=f"r={radius:.2f}")
ax.axhline(0, color="0.75", lw=0.8)
ax.set_xlabel("radial tangent component")
ax.set_ylabel("angular tangent component")
ax.set_title("Unit-ish metric ellipses in polar coordinates")
ax.legend()
fig2 = save_matplotlib(fig, chapter, "polar-metric-ellipses.png")
display_artifact(fig2)


## Flatness, Autoparallel Curves, and Embedded Curvature

Flatness is a statement about a connection, not about how a drawing looks in three-dimensional space. A cylinder is the standard sanity check: it looks curved in space, but a wrapped straight line is intrinsically straight. The figure unwraps a line from a flat rectangle and wraps it onto a cylinder. The line bends as an ambient curve, yet it remains the same intrinsic path in the unwrapped coordinates. This is the distinction the later statistical chapters exploit when exponential coordinates make some families straight and mixture coordinates make others straight.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

t = np.linspace(0, 2.8 * np.pi, 240)
z = np.linspace(-1.0, 1.0, t.size)
x = np.cos(t)
y = np.sin(t)
fig = plt.figure(figsize=(8.2, 4.4))
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(t, z, color="#1f77b4", lw=2.5)
ax1.set_xlabel("unwrapped angle")
ax1.set_ylabel("height")
ax1.set_title("flat chart line")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
uu = np.linspace(0, 2 * np.pi, 40)
zz = np.linspace(-1.1, 1.1, 20)
UU, ZZ = np.meshgrid(uu, zz)
ax2.plot_surface(np.cos(UU), np.sin(UU), ZZ, color="#dbe9f6", alpha=0.45, linewidth=0)
ax2.plot(x, y, z, color="#d62728", lw=2.5)
ax2.set_title("same path after embedding")
ax2.set_axis_off()
fig3 = save_matplotlib(fig, chapter, "flat-chart-wrapped-autoparallel.png")
display_artifact(fig3)


## Applied Lab: Connection Terms as Error Detectors

A useful way to debug a connection is to ask whether it catches the false acceleration created by a bad coordinate system. In polar coordinates, a point moving around a circle with constant angular speed has nonzero ordinary second derivative in the ambient plane, but in the coordinate chart the radial coordinate is constant. The connection contributes the missing radial term. If a numerical implementation forgets the `Gamma^r_{theta theta}` term, it will claim that circular motion has the wrong intrinsic acceleration. The final check records the symbolic Christoffel values and the generated artifact sizes.


For **01 Elementary Differential Geometry**, run the lab by naming the exact object being varied, the invariant being protected, and the hypothesis whose loss would break the conclusion. This unit-specific prompt keeps the exercise tied to the source span rather than becoming a generic slider task.

In [ ]:
omega = 0.7
radius = 1.3
coordinate_accel = np.array([0.0, 0.0])
connection_accel = np.array([float(Gamma[(0, 1, 1)].subs(r, radius)) * omega**2, 0.0])
radial_intrinsic = coordinate_accel[0] + connection_accel[0]
assert radial_intrinsic < 0

final_sanity = {
    "source_span": "printed pp. 1-24; physical DjVu pp. 10-33",
    "christoffel": {
        "Gamma_r_thetatheta": str(Gamma[(0, 1, 1)]),
        "Gamma_theta_rtheta": str(Gamma[(1, 0, 1)]),
    },
    "circular_connection_radial_term": radial_intrinsic,
}
check_path = save_json(final_sanity, chapter, "final_sanity.json")
sizes = require_artifacts([fig1, fig2, fig3, check_path])
final_sanity["artifact_sizes"] = sizes
save_json(final_sanity, chapter, "final_sanity.json")
assert len(sizes) == 4


## Source-Specific Inspection Notes

This enrichment note is specific to **01 Elementary Differential Geometry**. Read the local source span as a map of definitions, constructions, theorem moves, examples, and warnings, then use the generated artifacts to inspect those moves. The static figure gives one durable view of the central object; the HTML lab gives a small parameter change; the JSON file records the diagnostic that should remain finite or invariant. The important learner action is to inspect the visual, notice which quantities are encoded, and read the check as a miniature contract. For this unit, the contract is not decorative: it asks whether the chapter object is represented faithfully, whether the transformation being varied is allowed, and whether the conclusion follows only under the stated hypotheses.

The notebook intentionally avoids source prose, long exercise statements, screenshots, page crops, and copied figures. It uses printed pages and PDF pages only as source orientation. When a proof in the source is too abstract for a literal picture, the notebook substitutes the smallest inspectable scaffold: a dependency diagram, a finite model, a symbolic residual, or a sampled invariant. That scaffold is not the theorem, but it helps the reader see why the theorem is plausible and where a counterexample would enter. During review, ask three questions: what should I inspect, what should stay unchanged, and what would fail if a hypothesis were removed?

For **01 Elementary Differential Geometry**, extend the lab by adding one additional sample case. Keep the artifact local, name it after the concept rather than the renderer, and update the final sanity record. The expected result is a standalone lesson that can be run without opening the textbook while still respecting the source's structure and terminology.


In [ ]:
def assert_artifact(path):
    path = Path(path)
    assert path.exists(), f"missing artifact: {path}"
    assert path.stat().st_size > 20, f"tiny artifact: {path}"

# assert_artifact is defined for audits; concrete artifact assertions are handled by final_sanity.


Takeaways. A manifold is locally coordinatized, but its geometry is not the coordinate grid itself. Metrics measure tangent vectors, affine connections compare tangent vectors, autoparallel curves depend on the chosen connection, and curvature records failures of global flattening. These ideas become statistical in the next chapter: probability distributions form manifolds, likelihood derivatives become tangent vectors, and the Fisher metric supplies the local ruler.
